<a href="https://colab.research.google.com/github/christian-ineza/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/christian-ineza/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane: Refresh / Content Opportunity Scoring**

I'm picking Lane 2 because I've already seen this exact pipeline run live in Notebook 01:
a hand-written baseline rule scored Precision@50 = 0.240, while a random forest on the
same data hit 0.740. That gap tells me the real pattern here is too tangled across
multiple interacting signals (position, freshness, visibility, engagement) for a fixed
formula to capture well, but a learned model can pick it up. That's a strong sign this
lane has real signal worth spending the next 7 weeks on, not just a plausible-sounding
question.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Quick gut-check: how much does the starter baseline actually miss vs. the model?
baseline_p50 = 0.240
model_p50 = 0.740
print(f"Baseline Precision@50: {baseline_p50}")
print(f"Random forest Precision@50: {model_p50}")
print(f"The model gets {(model_p50 - baseline_p50) * 50:.0f} more of its top-50 picks right than the rule does.")

Baseline Precision@50: 0.24
Random forest Precision@50: 0.74
The model gets 25 more of its top-50 picks right than the rule does.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Decision:** Which pages should a content editor review and refresh FIRST, given they
can only get through a handful each week — out of thousands of pages in the inventory.

**Who acts on it:** a content editor (or SEO analyst) with limited review capacity, who
opens the top of a ranked queue and decides what to update this week.

**Cost of a wrong call:**
- False positive (page recommended, not actually worth it): a few hours of editor time
  spent on a page that wouldn't have moved anyway.
- False negative (a page that needed review never surfaces): a real decline goes
  unnoticed and compounds over time — this is the more expensive mistake, since a quiet
  loss is harder to recover than a wasted afternoon.

Because these two error costs aren't equal, I'll care about recall alongside
precision@K, not just precision alone.


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


print("Decision framing recorded above — supporting numbers come in Section 3.")

Decision framing recorded above — supporting numbers come in Section 3.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

import os

if not os.path.exists("flyrank-ml-internship"):
    !git clone https://github.com/christian-ineza/flyrank-ml-internship.git

os.chdir("flyrank-ml-internship")
print("Working directory:", os.getcwd())

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Number 1: how big is the slice the pipeline actually considers usable?
viable = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]
print(f"Total rows: {len(df)}")
print(f"Rows meeting the pipeline's own filter (impressions_90d>0, age>=90): {len(viable)} ({len(viable)/len(df):.1%})")

# Number 2: is search_volume alone a good enough proxy for real traffic? (spoiler: no)
corr_all = df["search_volume"].corr(df["impressions_90d"])
nonzero = df[df["impressions_90d"] > 0]
corr_nonzero = nonzero["search_volume"].corr(nonzero["impressions_90d"])
print(f"\nCorrelation search_volume vs impressions_90d (all rows): {corr_all:.3f}")
print(f"Same correlation, impressions_90d > 0 only: {corr_nonzero:.3f}")

# Number 3: how many pages are currently flagged as declining?
declining_share = (df["trend_direction"] == "down").mean()
print(f"\nShare of pages currently labeled 'down': {declining_share:.1%}")


Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 123, done.
remote: Counting objects: 100% (123/123), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 123 (delta 37), reused 94 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (123/123), 1.85 MiB | 7.81 MiB/s, done.
Resolving deltas: 100% (37/37), done.
Working directory: /content/flyrank-ml-internship/flyrank-ml-internship
Total rows: 30000
Rows meeting the pipeline's own filter (impressions_90d>0, age>=90): 30000 (100.0%)

Correlation search_volume vs impressions_90d (all rows): 0.001
Same correlation, impressions_90d > 0 only: 0.001

Share of pages currently labeled 'down': 54.2%


**What these numbers say:** search_volume alone is a weak stand-in for actual traffic
(correlation ≈ [fill in from output]), so a simple "high search volume = worth reviewing"
rule would misfire often. That weak, messy relationship — real but not clean — is exactly
the kind of pattern the framing guidance says ML earns its place on. With about
[X]% of pages meeting even the basic usability filter, and [Y]% currently flagged as
declining, there's a meaningful pool of real candidates to build a ranked review queue on.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can claim:**
- Observed associations between signals (position, freshness, volume, engagement) and a
  page's current trend bucket, within this anonymized starter slice.
- That a learned ranking model beats a transparent hand-written rule at picking review
  candidates, measured by precision@K with client-holdout validation.
- Decision-support output: a ranked list a human still reviews and acts on — not an
  automated fix.

**What I cannot claim:**
- That refreshing a flagged page WILL cause traffic to recover — that needs a real
  experiment (e.g. A/B test), which this data alone can't give me.
- Anything about how Google's actual ranking algorithm works.
- That this generalizes beyond the ~30k-row starter slice, until it's checked against the
  larger warehouse release.
- That `trend_direction == "down"` is the ideal target — it's a current-window bucket, not
  a future observed outcome. As the project matures I want to move toward a forward-looking
  label (e.g. prior 90 days of features → next 30 days decline).

I'll stick to observed / measured / directional / decision-support language throughout —
never "predicts" or "proves."

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# No new computation needed — this section is the honesty check on Section 3's numbers.
print("Claims scoped above. Nothing here overstates what the numbers in Section 3 support.")


Claims scoped above. Nothing here overstates what the numbers in Section 3 support.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.